In [1]:
import os
from docx import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from huggingface_hub import login


In [2]:
from huggingface_hub import login
login()


In [3]:
def load_all_docx(folder_path):
    all_texts = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            file_path = os.path.join(folder_path, filename)
            doc = Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
            all_texts.append(text)
    return all_texts

In [4]:

folder_path = r"C:\USDA\dataset"
texts = load_all_docx(folder_path)


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = splitter.create_documents(texts)

print(f" Loaded and split {len(documents)} document chunks.")


 Loaded and split 73 document chunks.


In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(documents, embedding_model)
db.save_local("faiss_hf_index")

print("✅ FAISS index created and saved locally.")


C:\Users\axi034\AppData\Local\Temp\ipykernel_8236\7504430.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


✅ FAISS index created and saved locally.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\axi034\AppData\Local\anaconda3\envs\rag-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axi034\.cache\huggingface\hub\models--microsoft--phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


In [9]:
from langchain.llms import HuggingFacePipeline
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [10]:
llm = HuggingFacePipeline(pipeline=pipe)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

C:\Users\axi034\AppData\Local\Temp\ipykernel_8236\2254101526.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [11]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

C:\Users\axi034\AppData\Local\Temp\ipykernel_8236\2971518456.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [12]:
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    output_key="answer"
)

In [ ]:
print("🤖 Chatbot is ready. Type 'exit' to quit.\n")

while True:
    query = input("📝 You: ")
    if query.lower() in ["exit", "quit", "bye"]:
        print("👋 Chatbot: Goodbye!")
        break

    result = rag_chain({"question": query})
    print(f"🤖 Chatbot: {result['answer']}\n")

🤖 Chatbot is ready. Type 'exit' to quit.



📝 You:  What is the Asian citrus psyllid and why is it important?


C:\Users\axi034\AppData\Local\Temp\ipykernel_8236\3297137473.py:9: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = rag_chain({"question": query})


🤖 Chatbot: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

This comprehensive report aims to provide an expert-level review of the Asian citrus psyllid and Huanglongbing. It delves into the intricate biology of the psyllid, the pathology and profound economic consequences of the disease, and the current integrated management strategies employed to combat this threat. Furthermore, the report compiles and details available datasets and resources, including critical image datasets, which are indispensable for research and control efforts. Finally, it explores the promising avenues of ongoing research and future prospects for ensuring the long-term sustainability and protection of the global citrus industry.
III. Biology and Identification of the Asian Citrus Psyllid (Diaphorina citri)

III. Biology and Identification of the Asian Citrus Psyllid (Diaphorina citri)
The Asian c

📝 You:  What disease does ACP transmit?
